# TP complet — Recalage d'images rétiniennes FIRE (version Kaggle corrigée)

**Objectif :** aligner une photographie du fond d'œil **mobile** sur une image **fixe**, analyser la transformation et le **flot optique résiduel**, puis mesurer la qualité à l'aide des **landmarks manuels**.

**Organisation :** configuration → identification de la base → primitives SIFT/ORB → RANSAC → transformation globale → comparaisons de similarité → visualisations des landmarks → flot optique → benchmark S / P / A → graphiques, export et conclusion.

> **Important :** les points de vérité terrain (*ground truth*) servent **exclusivement à l'évaluation**. Ils n'entrent ni dans l'appariement SIFT ni dans l'estimation RANSAC. Cela évite la fuite des données de test.

**Sources :** [FIRE officiel](https://projects.ics.forth.gr/cvrl/fire/) · [Dataset Kaggle](https://www.kaggle.com/datasets/andrewmvd/fundus-image-registration) · Hernandez-Matas et al. (2017), DOI `10.35119/maio.v1i4.42`.

## 0 — Préparation dans Kaggle

1. Sur la page du notebook Kaggle, clique sur **Add Input**, cherche `fundus-image-registration`, puis ajoute **Retina Fundus Image Registration** (Larxel).
2. Vérifie que la source est affichée dans **Input**. L'emplacement exact peut varier : **ne le saisis pas manuellement**.
3. Lance **Run All** à partir du haut. Toutes les fonctions de ce notebook sont incluses ; **aucun module `fire_registration.py` externe** n'est nécessaire.
4. La section **Détection FIRE** affiche un message explicite et une liste des fichiers trouvés si l'input n'a pas été correctement rattaché. Elle ne saute plus silencieusement les expériences.

Le dataset FIRE officiel contient **129 images / 134 paires**, réparties entre **S (71)**, **P (49)** et **A (14)** ; chaque paire possède une annotation `control_points_<identifiant>_1_2.txt`. Les images d'origine mesurent **2912 × 2912** pixels. Les calculs sont effectués sur des copies réduites, mais les erreurs TRE sont exprimées dans le **repère original en pixels**.

Les bibliothèques `opencv-python`, `numpy`, `matplotlib`, `pandas` sont généralement disponibles sur Kaggle. En cas de module manquant seulement : `!pip install opencv-contrib-python pandas matplotlib`.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
from IPython.display import display
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# PARAMÈTRES — modifiables
DATA_DIR = None         # None = détecte automatiquement le dataset sous /kaggle/input
DEMO_SIDE = 850         # images visualisées, en pixels sur le plus grand côté
BENCH_SIDE = 900        # redimensionnement pour le benchmark (750 plus rapide)
MAX_PAIRS = None        # None = toutes les 134 paires; 10 pour un test rapide
THRESHOLD_PX = 50.0    # seuil de succès pédagogique, non officiel FIRE
BENCH_MODELS = ('similarity', 'affine', 'homography')
RESULT_DIR = Path('/kaggle/working/FIRE_results') if Path('/kaggle/working').is_dir() else Path.cwd()/'FIRE_results'

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.dpi'] = 105
print('OpenCV :', cv2.__version__)
print('Détecteur par défaut :', 'SIFT' if hasattr(cv2, 'SIFT_create') else 'ORB')
print('Dossier des résultats :', RESULT_DIR)

## 1 — Chargement et vérification des paires

Les deux images ont pour noms `<identifiant>_1.jpg` (**fixe**) et `<identifiant>_2.jpg` (**mobile**). Le texte d'annotation comprend des lignes de la forme `(x_ref, y_ref, x_mobile, y_mobile)`.

**Correction apportée :** au lieu de chercher *uniquement* dans un dossier codé en dur (`/kaggle/input/fundus-image-registration`), le chargeur examine **tous les dossiers attachés sous `/kaggle/input`**, cherche récursivement les images et les annotations, et sélectionne le jeu possédant le plus de paires valides. Aucune récupération réseau n'est requise.

In [ ]:
@dataclass(frozen=True)
class FirePair:
    pair_id: str
    category: str
    fixed_path: Path
    moving_path: Path
    ground_truth_path: Path


def find_fire_pairs(root: str | Path) -> list[FirePair]:
    """Find pairs regardless of where FIRE/Images and FIRE/Ground Truth reside."""
    root = Path(root).expanduser()
    if not root.exists():
        raise FileNotFoundError(f"Chemin introuvable : {root}")
    imgs = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}:
            imgs.setdefault(p.stem.lower(), []).append(p)
    result = []
    pattern = re.compile(r'^control_points_(.+)_1_2$', re.IGNORECASE)
    for gt in sorted(root.rglob('*')):
        if not gt.is_file() or gt.suffix.lower() != '.txt':
            continue
        match = pattern.match(gt.stem)
        if not match:
            continue
        pair_id = match.group(1)
        fixed_options = imgs.get(f'{pair_id}_1'.lower(), [])
        moving_options = imgs.get(f'{pair_id}_2'.lower(), [])
        if not fixed_options or not moving_options:
            continue
        # If there are several copied images, prefer the closest files.
        fixed = min(fixed_options, key=lambda p: len(p.parts))
        moving = min(moving_options, key=lambda p: len(p.parts))
        category = pair_id[0].upper() if pair_id[0].upper() in 'SAP' else 'OTHER'
        result.append(FirePair(pair_id, category, fixed, moving, gt))
    return sorted(result, key=lambda p: (p.category, p.pair_id))


def discover_fire_dataset(data_dir=None):
    """Find real FIRE pairs on Kaggle regardless of the attached input slug.

    Never search synthetic_demo in /kaggle/working: one demo pair is not FIRE.
    Returns (pairs, selected_root, diagnostics); it does not download data.
    """
    candidates = []
    if data_dir not in (None, ''):
        candidates.append(Path(data_dir).expanduser())
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        candidates.extend(sorted(p for p in kaggle_input.iterdir() if p.is_dir()))
    candidates.extend([Path.cwd() / 'data', Path.cwd() / 'FIRE', Path('/mnt/data/FIRE')])

    best_pairs = []
    best_root = None
    log = []
    seen = set()
    for root in candidates:
        k = str(root.resolve()) if root.exists() else str(root)
        if k in seen:
            continue
        seen.add(k)
        if not root.is_dir():
            log.append(f'{root} : chemin absent')
            continue
        try:
            found = find_fire_pairs(root)
            log.append(f'{root} : {len(found)} paire(s) annotée(s)')
            if len(found) > len(best_pairs):
                best_pairs, best_root = found, root
        except Exception as exc:
            log.append(f'{root} : erreur ({type(exc).__name__}: {exc})')
    return best_pairs, best_root, log


def fire_input_diagnostics():
    """Explain missing input instead of silently skipping all experiments."""
    root = Path('/kaggle/input')
    if not root.is_dir():
        return 'Pas de /kaggle/input : exécution hors Kaggle ou jeu de données absent.'
    datasets = sorted(p for p in root.iterdir() if p.is_dir())
    if not datasets:
        return 'Aucun dataset monté dans /kaggle/input. Sur Kaggle : Add Input → FIRE → +.'
    info = []
    for d in datasets[:12]:
        imgs = list(d.rglob('*.jpg'))[:3]
        gt = list(d.rglob('control_points_*.txt'))[:3]
        info.append(f'{d.name} | exemples JPG : {[x.relative_to(d).as_posix() for x in imgs]} | '                    f'exemples GT : {[x.relative_to(d).as_posix() for x in gt]}')
    return '\n'.join(info)


def read_pair(pair: FirePair, max_side: int = 950):
    """Load two full images, resize independently and scale landmarks accordingly."""
    fixed_full = cv2.imread(str(pair.fixed_path), cv2.IMREAD_COLOR)
    moving_full = cv2.imread(str(pair.moving_path), cv2.IMREAD_COLOR)
    if fixed_full is None or moving_full is None:
        raise ValueError(f"Impossible de lire les images : {pair.pair_id}")
    gt = np.loadtxt(pair.ground_truth_path, dtype=np.float64)
    gt = np.atleast_2d(gt)
    if gt.shape[1] != 4 or len(gt) < 3:
        raise ValueError(f"Landmarks mal formés pour {pair.pair_id}: {gt.shape}")

    def resize(img):
        height, width = img.shape[:2]
        fac = min(1.0, max_side / max(height, width))
        dst_w = max(1, int(round(width * fac)))
        dst_h = max(1, int(round(height * fac)))
        small = cv2.resize(img, (dst_w, dst_h), interpolation=cv2.INTER_AREA) if fac < 1 else img
        # Use actual integer-resizing ratios for geometric consistency.
        sx = dst_w / width
        sy = dst_h / height
        return small, np.diag([sx, sy, 1.0]).astype(np.float64)

    fixed, sf = resize(fixed_full)
    moving, sm = resize(moving_full)
    # FIRE: x_ref,y_ref,x_test,y_test. Use annotations ONLY for evaluation.
    ref_gt_full = gt[:, :2].copy()
    mov_gt_full = gt[:, 2:].copy()
    return fixed, moving, sf, sm, ref_gt_full, mov_gt_full

## 2 — Détection et mise en correspondance de primitives

Nous utilisons le canal **vert**, plus contrasté pour les vaisseaux, suivi de **CLAHE**. Les points sont calculés par **SIFT**, avec repli sur ORB si SIFT est indisponible. L'appariement utilise le rapport de Lowe avant estimation de la transformation par **RANSAC**.

Trois modèles globaux sont comparés :

| Modèle | Ce qu'il peut représenter | Limite principale |
|---|---|---|
| **Similarité** | Translation, rotation, échelle uniforme | Ni cisaillement ni déformation locale |
| **Affine** | Translation, rotation, échelles, cisaillement | Conserve le parallélisme |
| **Homographie** | Transformation projective | Ne décrit pas une déformation anatomique élastique |

Le point d'arrivée est toujours **l'image fixe** : `H @ coordonnées_mobile → coordonnées_fixe`.

In [ ]:
def preprocess_for_features(bgr: np.ndarray) -> np.ndarray:
    """CLAHE on green channel: vascular structures have useful contrast there."""
    green = bgr[:, :, 1]
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(green)


def match_features(fixed: np.ndarray, moving: np.ndarray, ratio: float = 0.78):
    fgray = preprocess_for_features(fixed)
    mgray = preprocess_for_features(moving)
    if hasattr(cv2, 'SIFT_create'):
        detector = cv2.SIFT_create(nfeatures=5000, contrastThreshold=0.012)
        norm = cv2.NORM_L2
        detector_name = 'SIFT'
    else:
        detector = cv2.ORB_create(nfeatures=6000, fastThreshold=8)
        norm = cv2.NORM_HAMMING
        detector_name = 'ORB'
    kf, df = detector.detectAndCompute(fgray, None)
    km, dm = detector.detectAndCompute(mgray, None)
    if df is None or dm is None or len(df) < 2 or len(dm) < 2:
        return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32), 0, detector_name
    matcher = cv2.BFMatcher(norm)
    matches = matcher.knnMatch(dm, df, k=2)  # Moving -> fixed: warp M into F.
    good = [pair[0] for pair in matches if len(pair) == 2 and pair[0].distance < ratio * pair[1].distance]
    if not good:
        return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32), 0, detector_name
    points_m = np.float32([km[m.queryIdx].pt for m in good])
    points_f = np.float32([kf[m.trainIdx].pt for m in good])
    return points_m, points_f, len(good), detector_name


def estimate_transform(src: np.ndarray, dst: np.ndarray, model: str, seed: int = 42):
    """Return a moving->fixed 3x3 matrix in processed pixel coordinates."""
    min_matches = 4 if model == 'homography' else 3
    if len(src) < min_matches:
        return None, 0
    cv2.setRNGSeed(seed)
    try:
        if model == 'similarity':
            affine, inliers = cv2.estimateAffinePartial2D(
                src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.5,
                maxIters=5000, confidence=0.995, refineIters=15)
        elif model == 'affine':
            affine, inliers = cv2.estimateAffine2D(
                src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.5,
                maxIters=5000, confidence=0.995, refineIters=15)
        elif model == 'homography':
            h, inliers = cv2.findHomography(
                src, dst, cv2.RANSAC, 4.0, maxIters=5000, confidence=0.995)
            if h is None or not np.isfinite(h).all():
                return None, 0
            return h.astype(np.float64), int(inliers.sum()) if inliers is not None else 0
        else:
            raise ValueError(f"Modèle non reconnu: {model}")
    except cv2.error:
        return None, 0
    if affine is None or not np.isfinite(affine).all():
        return None, 0
    return np.vstack([affine, [0.0, 0.0, 1.0]]), int(inliers.sum()) if inliers is not None else 0


def warp_to_fixed(moving: np.ndarray, fixed: np.ndarray, h_proc: np.ndarray):
    w, h = fixed.shape[1], fixed.shape[0]
    return cv2.warpPerspective(moving, h_proc, (w, h), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

## 3 — Critères de similarité et de recouvrement

Nous calculons les indicateurs sur la **même zone valide** quand nous comparons l'avant et l'après :

- **MSE** : erreur quadratique moyenne, à **minimiser** (le SSD est `MSE × nombre_de_pixels`).
- **NCC / Pearson** : corrélation linéaire, à **maximiser** vers 1.
- **NMI** : information mutuelle normalisée, qui exploite l'histogramme conjoint et est moins sensible aux changements d'intensité.

Le fond noir hors rétine n'est pas inclus dans les métriques. Une petite zone commune peut fausser les comparaisons ; c'est pourquoi on affiche aussi le **taux de recouvrement**.

In [ ]:
def valid_retina_mask(bgr: np.ndarray) -> np.ndarray:
    """Approximate nonblack fundus ROI; not a clinical segmentation."""
    return (np.max(bgr, axis=2) > 18).astype(np.uint8)


def paired_overlap_mask(fixed: np.ndarray, moving: np.ndarray, h_proc: np.ndarray):
    roi_f = valid_retina_mask(fixed).astype(np.uint8)
    roi_m = valid_retina_mask(moving).astype(np.uint8)
    warped_roi = cv2.warpPerspective(roi_m, h_proc, (fixed.shape[1], fixed.shape[0]),
                                     flags=cv2.INTER_NEAREST)
    return (roi_f > 0) & (warped_roi > 0), float(((roi_f > 0) & (warped_roi > 0)).sum() / max(int(roi_f.sum()), 1))


def similarity_metrics(fixed: np.ndarray, registered: np.ndarray, overlap: np.ndarray, bins: int = 32):
    """MSE, NCC and NMI on identical overlapping valid-pixel coordinates."""
    if overlap.sum() < 500:
        return {'mse': np.nan, 'ncc': np.nan, 'nmi': np.nan}
    # Same green channel for each method; green channel highlights vessels.
    f = fixed[:, :, 1][overlap].astype(np.float64)
    m = registered[:, :, 1][overlap].astype(np.float64)
    mse = float(np.mean((f - m) ** 2))
    sf, sm = f.std(), m.std()
    ncc = float(np.mean(((f - f.mean()) / sf) * ((m - m.mean()) / sm))) if sf > 1e-12 and sm > 1e-12 else np.nan
    hist, _, _ = np.histogram2d(f, m, bins=bins, range=[[0, 256], [0, 256]])
    p = hist / max(hist.sum(), 1)
    px, py = p.sum(axis=1), p.sum(axis=0)
    def entropy(prob):
        nz = prob > 0
        return float(-np.sum(prob[nz] * np.log(prob[nz])))
    hx, hy, hxy = entropy(px), entropy(py), entropy(p)
    mi = max(0.0, hx + hy - hxy)
    nmi = float(2.0 * mi / (hx + hy)) if hx + hy > 0 else np.nan
    return {'mse': mse, 'ncc': ncc, 'nmi': nmi}


def transform_points(pts: np.ndarray, h: np.ndarray):
    transformed = cv2.perspectiveTransform(np.asarray(pts, np.float64).reshape(-1, 1, 2),
                                            np.asarray(h, np.float64)).reshape(-1, 2)
    return transformed

## 4 — Évaluation : erreur des landmarks (TRE)

Soient les landmarks mobiles `pᵢ` et les landmarks de référence `qᵢ`. Après estimation d'une transformation globale `H` **sans lire les landmarks**, on mesure :

\[\operatorname{TRE}(H)=\frac1N\sum_{i=1}^{N}\|H(p_i)-q_i\|_2\]

La TRE est calculée en **pixels de l'image originale** en convertissant `H` depuis la résolution réduite. Les échecs RANSAC sont marqués `failed`, non remplacés silencieusement par une transformation identité. Le **taux de succès** est calculé sur **toutes les paires**, y compris les échecs.

In [ ]:
def score_with_matches(pair: FirePair, model: str, fixed: np.ndarray,
                       moving: np.ndarray, sf: np.ndarray, sm: np.ndarray,
                       ref_gt: np.ndarray, mov_gt: np.ndarray,
                       pts_m: np.ndarray, pts_f: np.ndarray,
                       n_matches: int, detector: str):
    """Score a fit: landmarks enter here only AFTER the transform is estimated."""
    h_proc, n_inliers = estimate_transform(pts_m, pts_f, model)
    before_tre = float(np.mean(np.linalg.norm(mov_gt - ref_gt, axis=1)))
    record = dict(pair_id=pair.pair_id, category=pair.category, model=model,
                  detector=detector, status='failed', n_matches=n_matches,
                  n_inliers=n_inliers, tre_before_px=before_tre, tre_after_px=np.nan,
                  tre_median_px=np.nan, n_landmarks=len(ref_gt), overlap_fraction=np.nan,
                  mse=np.nan, ncc=np.nan, nmi=np.nan)
    if h_proc is None or n_inliers < (4 if model == 'homography' else 3):
        return record, None
    # Undo processing resize to assess TRE IN ORIGINAL FIRE PIXELS.
    h_full = np.linalg.inv(sf) @ h_proc @ sm
    if not np.isfinite(h_full).all():
        return record, None
    pred_gt = transform_points(mov_gt, h_full)
    if not np.isfinite(pred_gt).all():
        return record, None
    errors = np.linalg.norm(pred_gt - ref_gt, axis=1)
    warped = warp_to_fixed(moving, fixed, h_proc)
    overlap, coverage = paired_overlap_mask(fixed, moving, h_proc)
    sims = similarity_metrics(fixed, warped, overlap)
    record.update(status='ok', tre_after_px=float(np.mean(errors)),
                  tre_median_px=float(np.median(errors)), overlap_fraction=coverage, **sims)
    detail = dict(fixed=fixed, moving=moving, warped=warped, h_proc=h_proc,
                  h_full=h_full, gt_fixed=ref_gt, gt_moving=mov_gt,
                  overlap=overlap, matched_moving=pts_m, matched_fixed=pts_f,
                  predicted_gt=pred_gt, errors=errors)
    return record, detail


def evaluate_pair(pair: FirePair, model: str = 'affine', max_side: int = 950):
    """Estimate using SIFT/ORB only; evaluate against unobserved annotated GT."""
    fixed, moving, sf, sm, ref_gt, mov_gt = read_pair(pair, max_side)
    pts_m, pts_f, n_matches, detector = match_features(fixed, moving)
    return score_with_matches(pair, model, fixed, moving, sf, sm,
                              ref_gt, mov_gt, pts_m, pts_f, n_matches, detector)


def evaluate_dataset(pairs: list[FirePair], models=('similarity', 'affine', 'homography'),
                     max_side: int = 950, max_pairs: Optional[int] = None, verbose=True):
    """Load each pair and detect/match features only ONCE for all models."""
    selected = pairs if max_pairs is None else pairs[:max_pairs]
    records = []
    for i, pair in enumerate(selected, start=1):
        try:
            fixed, moving, sf, sm, ref_gt, mov_gt = read_pair(pair, max_side)
            pts_m, pts_f, n_matches, detector = match_features(fixed, moving)
            for model in models:
                try:
                    record, _ = score_with_matches(
                        pair, model, fixed, moving, sf, sm,
                        ref_gt, mov_gt, pts_m, pts_f, n_matches, detector)
                except Exception as ex:
                    record = dict(pair_id=pair.pair_id, category=pair.category, model=model,
                                  status=f'error:{str(ex)[:80]}')
                records.append(record)
        except Exception as ex:
            for model in models:
                records.append(dict(pair_id=pair.pair_id, category=pair.category, model=model,
                                    status=f'error:{str(ex)[:80]}'))
        if verbose and (i % 10 == 0 or i == len(selected)):
            print(f'Paires traitées : {i}/{len(selected)}')
    return pd.DataFrame.from_records(records)


def summarize_results(results: pd.DataFrame, threshold_px: float = 50.0):
    """Success at threshold uses ALL pairs as denominator, including failures."""
    r = results.copy()
    if r.empty:
        return pd.DataFrame()
    r['successful_at_threshold'] = (r['status'] == 'ok') & (r['tre_after_px'] < threshold_px)
    table = r.groupby(['category', 'model'], dropna=False).agg(
        total_pairs=('pair_id', 'count'), fitted_pairs=('status', lambda x: int((x == 'ok').sum())),
        success_rate=('successful_at_threshold', 'mean'),
        mean_tre_px=('tre_after_px', 'mean'), median_tre_px=('tre_after_px', 'median'),
        mean_ncc=('ncc', 'mean'), mean_nmi=('nmi', 'mean'),
        mean_mse=('mse', 'mean'), mean_coverage=('overlap_fraction', 'mean')).reset_index()
    return table


def compare_similarity_before_after(pair: FirePair, model='affine', max_side=950):
    """Fairly compare metrics on the SAME valid pixel coordinates before/after."""
    record, detail = evaluate_pair(pair, model, max_side)
    if detail is None:
        return None
    fixed, moving, warped = detail['fixed'], detail['moving'], detail['warped']
    if moving.shape[:2] != fixed.shape[:2]:
        moving = cv2.resize(moving, (fixed.shape[1], fixed.shape[0]))
    common = ((valid_retina_mask(fixed) > 0) &
              (valid_retina_mask(moving) > 0) &
              (valid_retina_mask(warped) > 0))
    return pd.DataFrame([
        dict(etape='avant', pixels=int(common.sum()), **similarity_metrics(fixed, moving, common)),
        dict(etape='apres', pixels=int(common.sum()), **similarity_metrics(fixed, warped, common)),
    ])

## 5 — Figures : correspondances, avant/après, points de contrôle, flot optique

Le flot optique résiduel utilise **Farnebäck** après recalage global. Il illustre les déplacements locaux, mais il ne prouve **pas** l'existence d'un changement anatomique : l'éclairage, le bruit ou les erreurs de transformation peuvent également produire du flot.

In [ ]:
def select_demo_pair(pairs: list[FirePair], category='S', model='affine',
                     max_side=850, max_attempts=8):
    """Try multiple examples so that a poor first sample won't break the demo."""
    candidates = [p for p in pairs if p.category.upper() == category.upper()]
    for pair in candidates[:max_attempts]:
        try:
            record, detail = evaluate_pair(pair, model=model, max_side=max_side)
        except Exception:
            continue
        if detail is not None and np.isfinite(record['tre_after_px']):
            return pair
    return candidates[0] if candidates else None


def plot_feature_matches(pair: FirePair, max_side=900, max_draw=60):
    """Visualize SIFT/ORB point matches; landmarks are not used here."""
    fixed, moving, *_ = read_pair(pair, max_side=max_side)
    img1 = preprocess_for_features(moving)
    img2 = preprocess_for_features(fixed)
    if hasattr(cv2, 'SIFT_create'):
        det = cv2.SIFT_create(nfeatures=3500, contrastThreshold=0.012)
        norm = cv2.NORM_L2
    else:
        det = cv2.ORB_create(nfeatures=4500, fastThreshold=8)
        norm = cv2.NORM_HAMMING
    kp1, des1 = det.detectAndCompute(img1, None)
    kp2, des2 = det.detectAndCompute(img2, None)
    if des1 is None or des2 is None:
        print('Descripteurs non détectés sur cette paire.')
        return
    raw = cv2.BFMatcher(norm).knnMatch(des1, des2, k=2)
    matches = [m for ls in raw if len(ls) == 2 for m, n in [ls] if m.distance < .78*n.distance]
    matches.sort(key=lambda x: x.distance)
    canvas = cv2.drawMatches(img1, kp1, img2, kp2, matches[:max_draw], None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(16, 6))
    plt.imshow(canvas, cmap='gray')
    plt.title(f'Correspondances SIFT/ORB — {pair.pair_id} — {len(matches)} matches')
    plt.axis('off')
    plt.show()


def show_registration(pair: FirePair, model='affine', max_side=950, figsize=(16, 9)):
    record, detail = evaluate_pair(pair, model, max_side)
    if detail is None:
        print(f"Échec du recalage {pair.pair_id} / {model}. Correspondances : {record['n_matches']}")
        return record
    fixed = cv2.cvtColor(detail['fixed'], cv2.COLOR_BGR2RGB)
    moving = cv2.cvtColor(detail['moving'], cv2.COLOR_BGR2RGB)
    warped = cv2.cvtColor(detail['warped'], cv2.COLOR_BGR2RGB)
    before_overlay = cv2.addWeighted(fixed, .5, cv2.resize(moving, (fixed.shape[1], fixed.shape[0])), .5, 0)
    after_overlay = cv2.addWeighted(fixed, .5, warped, .5, 0)
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    panels = [(fixed, 'Référence (fixe)'), (moving, 'Image à recaler (mobile)'),
              (warped, f'Image recalée ({model})'), (before_overlay, 'Superposition AVANT'),
              (after_overlay, 'Superposition APRÈS'), (cv2.absdiff(fixed, warped), 'Différence absolue (RGB)')]
    for ax, (img, title) in zip(axes.flat, panels):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f"FIRE {pair.pair_id} | TRE {record['tre_before_px']:.1f} → {record['tre_after_px']:.1f} px | "+
                 f"{record['n_inliers']} inliers", fontsize=13)
    fig.tight_layout()
    plt.show()
    return record


def plot_landmarks(pair: FirePair, model='affine', max_side=950):
    record, detail = evaluate_pair(pair, model, max_side)
    if detail is None:
        print('Transformation non estimée')
        return record
    sf = detail['fixed'].shape[1] / cv2.imread(str(pair.fixed_path)).shape[1]
    sy = detail['fixed'].shape[0] / cv2.imread(str(pair.fixed_path)).shape[0]
    ref = detail['gt_fixed'] * [sf, sy]
    predicted = detail['predicted_gt'] * [sf, sy]
    rgb = cv2.cvtColor(detail['fixed'], cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(rgb)
    ax.scatter(ref[:, 0], ref[:, 1], s=32, c='lime', marker='o', label='Landmarks référence')
    ax.scatter(predicted[:, 0], predicted[:, 1], s=35, c='red', marker='x', label='Landmarks après recalage')
    for a, b in zip(predicted, ref):
        ax.plot([a[0], b[0]], [a[1], b[1]], c='yellow', alpha=.8, linewidth=1)
    ax.legend(loc='lower left')
    ax.axis('off')
    ax.set_title(f'Erreur landmarks : {record["tre_after_px"]:.1f} pixels (résolution originale)')
    plt.show()
    return record


def visualize_optical_flow(pair: FirePair, model='affine', max_side=900, optical_side=440):
    """Teaching example: dense residual flow after global registration, not GT fit."""
    record, detail = evaluate_pair(pair, model, max_side)
    if detail is None:
        print('Recalage global échoué, flot résiduel non disponible.')
        return record
    fixed, warped = detail['fixed'], detail['warped']
    sc = min(1., optical_side/max(fixed.shape[:2]))
    shape = (int(fixed.shape[1]*sc), int(fixed.shape[0]*sc))
    fixed = cv2.resize(fixed, shape)
    warped = cv2.resize(warped, shape)
    fg = preprocess_for_features(fixed)
    wg = preprocess_for_features(warped)
    flow = cv2.calcOpticalFlowFarneback(fg, wg, None, .5, 3, 21, 4, 7, 1.5, 0)
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    hsv = np.zeros((*mag.shape, 3), np.uint8)
    hsv[..., 0] = np.uint8(ang*180/np.pi/2)
    hsv[..., 1] = 255
    hsv[..., 2] = np.uint8(np.clip(mag / max(float(np.quantile(mag, .98)), 1e-4)*255, 0, 255))
    flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    gy, gx = np.mgrid[0:flow.shape[0], 0:flow.shape[1]]
    # Inverse sampling: flow ref -> moving; sample moving at ref + flow.
    refined = cv2.remap(warped, (gx+flow[...,0]).astype(np.float32),
                        (gy+flow[...,1]).astype(np.float32), cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_CONSTANT)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, im, name in zip(axes,
                            [cv2.cvtColor(warped, cv2.COLOR_BGR2RGB), flow_rgb, cv2.cvtColor(refined, cv2.COLOR_BGR2RGB)],
                            ['Après recalage global', 'Champ optique résiduel (HSV)', 'Correction locale indicative']):
        ax.imshow(im);ax.set_title(name);ax.axis('off')
    plt.tight_layout();plt.show()
    print(f"Norme moyenne du flot résiduel : {np.mean(mag):.2f} px à résolution {shape}. "
          'Attention : l’éclairage et les erreurs de recalage affectent aussi ce flot.')
    return record

## 6 — Auto-test synthétique (faculté de vérifier même sans FIRE)

Nous créons une paire artificielle avec rotation, translation, changement d'échelle, dessin simplifié de vaisseaux et 10 points connus. Ce test vérifie la cohérence du sens des transformations et l'unité de la TRE. **Ce n'est pas une évaluation réelle sur FIRE**.

In [ ]:
def make_synthetic_fire(root: str | Path, shape: int = 700):
    """Create a fully controlled synthetic self-test, NOT real FIRE images."""
    root = Path(root)
    d = root/'FIRE'/'Images'; g = root/'FIRE'/'Ground Truth'
    d.mkdir(parents=True, exist_ok=True);g.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(12)
    img = np.zeros((shape, shape, 3), np.uint8)
    yy, xx = np.ogrid[:shape, :shape]
    cx, cy, radius = shape/2, shape/2, shape*.445
    inside = (xx-cx)**2+(yy-cy)**2 < radius**2
    grad = 1 - .36*((xx-cx)**2+(yy-cy)**2)/radius**2
    img[inside] = np.array([39, 91, 154], np.float64)*np.clip(grad[inside,None], .25, 1)
    # Synthetic optic disc and branching vasculature.
    disc = (int(shape*.35), int(shape*.48))
    cv2.circle(img, disc, int(shape*.065), (105, 178, 210), -1, cv2.LINE_AA)
    for i in range(70):
        start = (int(shape*(.25+rng.uniform(-.04, .07))), int(shape*(.47+rng.uniform(-.08,.08))))
        theta = rng.uniform(-np.pi, np.pi)
        rr = rng.uniform(.16,.42)*shape
        end = (int(np.clip(start[0]+rr*np.cos(theta), 0,shape-1)),
               int(np.clip(start[1]+rr*np.sin(theta), 0,shape-1)))
        mid = (int(.5*(start[0]+end[0])+rng.uniform(-.03,.03)*shape),
               int(.5*(start[1]+end[1])+rng.uniform(-.03,.03)*shape))
        thickness = max(1, int(rng.uniform(.002,.008)*shape))
        cv2.line(img,start,mid,(19,34,71),thickness,cv2.LINE_AA)
        cv2.line(img,mid,end,(19,34,71),max(1,thickness-1),cv2.LINE_AA)
    grain = rng.normal(0,3,img.shape)
    img = np.uint8(np.clip(img.astype(np.float64)+grain,0,255))
    # Moving source is a transformed version of ref; GT coordinates are known exactly.
    m = cv2.getRotationMatrix2D((cx,cy), 11, 1.04)
    m[0,2] += 27
    m[1,2] -= 23
    # Mapping reference -> moving for generating the source image.
    moving = cv2.warpAffine(img,m,(shape,shape),borderValue=(0,0,0))
    moving = np.uint8(np.clip(moving.astype(np.float32)*1.08,0,255))
    landmarks = np.array([(shape*x,shape*y) for x,y in
                          [(0.27,.39),(.35,.41),(.47,.33),(.54,.46),(.43,.58),(.62,.61),(.64,.42),(.32,.63),(.53,.71),(.70,.52)]],
                         np.float64)
    landmark_m = cv2.transform(landmarks[None,:,:],m)[0]
    cv2.imwrite(str(d/'s999_1.jpg'),img)
    cv2.imwrite(str(d/'s999_2.jpg'),moving)
    np.savetxt(g/'control_points_s999_1_2.txt',np.hstack([landmarks,landmark_m]),fmt='%.6f')
    return root

In [ ]:
test_root = make_synthetic_fire(RESULT_DIR / 'synthetic_demo', shape=700)
test_pairs = find_fire_pairs(test_root)
test_results = evaluate_dataset(test_pairs, models=BENCH_MODELS, max_side=500, verbose=False)
print('Contrôle synthétique :')
display(test_results[['model','status','n_matches','n_inliers','tre_before_px','tre_after_px']].round(3))
assert len(test_pairs) == 1
assert (test_results['status'] == 'ok').all(), 'Test synthétique échoué : vérifier la transformation.'
assert (test_results['tre_after_px'] < 3.0).all(), 'TRE synthétique excessive.'
print('✅ Auto-test synthétique validé (ne pas confondre avec FIRE).')

## 7 — Détecter **le vrai dataset FIRE** sous Kaggle

**Cette section remplace entièrement l'ancienne cellule `DATA_DIR = None` et les anciens blocs `if pairs: ... else: Section sautée`.** Elle cherche partout sous les répertoires d'entrée montés par Kaggle et affiche les noms des fichiers si les correspondances ne sont pas trouvées. Dans ce cas elle lève une **erreur explicative**, au lieu de prétendre que l'expérience a été effectuée.

In [ ]:
pairs, resolved_dir, search_log = discover_fire_dataset(DATA_DIR)
print('=== DIAGNOSTIC FIRE ===')
for line in search_log:
    print('  ', line)
print('=======================')

if not pairs:
    print('\nAucun couple JPG + annotations trouvé. Détails des inputs Kaggle :')
    print(fire_input_diagnostics())
    raise FileNotFoundError(
        'FIRE absent. Sur Kaggle : Add Input → Retina Fundus Image Registration '
        '(Larxel) → bouton +, puis Run All. Vérifier la présence de fichiers '
        'control_points_*.txt et de *_1.jpg / *_2.jpg.'
    )

DATA_DIR = str(resolved_dir)
counts = pd.Series([p.category for p in pairs]).value_counts().reindex(['S','P','A'], fill_value=0)
print(f'✅ FIRE détecté sous : {resolved_dir}')
print(f'✅ {len(pairs)} paires valides détectées sur 134 attendues')
print('Répartition des catégories :', counts.to_dict())
print('Exemple :', pairs[0].pair_id)
print('  FIXE :', pairs[0].fixed_path)
print('  MOBILE :', pairs[0].moving_path)
print('  GT :', pairs[0].ground_truth_path)
if len(pairs) != 134 or counts.to_dict() != {'S':71,'P':49,'A':14}:
    warnings.warn('Le nombre de paires/catégories diffère du FIRE officiel : vérifier le dataset attaché.')

display(pd.DataFrame([
    {'Paire':p.pair_id,'Catégorie':p.category,'Référence':p.fixed_path.name,
     'Mobile':p.moving_path.name,'Annotations':p.ground_truth_path.name}
    for p in pairs[:8]
]))

## 8 — Inspecter les images et les correspondances SIFT

Nous choisissons automatiquement une paire de catégorie **S** pour laquelle le recalage affine fonctionne si possible. Le choix s'effectue sur la réussite de l'appariement/estimation, et les landmarks sont uniquement affichés a posteriori. Les trois catégories seront évaluées ensuite par le benchmark complet.

In [ ]:
sample = select_demo_pair(pairs, category='S', model='affine', max_side=DEMO_SIDE)
if sample is None:
    sample = pairs[0]
print('Paire de démonstration :', sample.pair_id, '| catégorie :', sample.category)
fixed, moving, _, _, ref_points, mov_points = read_pair(sample, max_side=DEMO_SIDE)
fig, axes = plt.subplots(1, 2, figsize=(12,6))
axes[0].imshow(cv2.cvtColor(fixed,cv2.COLOR_BGR2RGB)); axes[0].set_title('Image fixe / référence')
axes[1].imshow(cv2.cvtColor(moving,cv2.COLOR_BGR2RGB)); axes[1].set_title('Image mobile / test')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Nombre de landmarks annotés (évaluation) : {len(ref_points)}')
plot_feature_matches(sample, max_side=DEMO_SIDE, max_draw=60)

## 9 — Recalage global : visualiser avant et après

On affiche la paire choisie, la superposition avant/après et la différence absolue. Puis, pour **S, A, P**, on tente une paire représentative par catégorie et on affiche explicitement l'échec lorsque RANSAC ne trouve pas de transformation.

In [ ]:
print('=== EXEMPLE S ===')
registration_s = show_registration(sample, model='affine', max_side=DEMO_SIDE)

for cat in ('A','P'):
    p = select_demo_pair(pairs, category=cat, model='affine', max_side=DEMO_SIDE, max_attempts=5)
    if p is None:
        print(f'Aucune paire trouvée en catégorie {cat}.')
        continue
    print(f'=== EXEMPLE {cat} : {p.pair_id} ===')
    show_registration(p, model='affine', max_side=DEMO_SIDE)

## 10 — Landmarks, comparaison MSE/NCC/NMI et matrice de corrélation

Les cercles verts indiquent les landmarks de référence ; les croix rouges les points mobiles **projetés**. Les segments jaunes matérialisent l'erreur. Les métriques **avant et après** sont calculées sur les **mêmes coordonnées de recouvrement**.

In [ ]:
landmark_result = plot_landmarks(sample, model='affine', max_side=DEMO_SIDE)
comparison = compare_similarity_before_after(sample, model='affine', max_side=DEMO_SIDE)
if comparison is not None:
    display(comparison.round(4))
    _, det = evaluate_pair(sample, model='affine', max_side=DEMO_SIDE)
    if det is not None:
        f, m, w = det['fixed'], det['moving'], det['warped']
        if m.shape[:2] != f.shape[:2]: m = cv2.resize(m, (f.shape[1],f.shape[0]))
        common = (valid_retina_mask(f).astype(bool) & valid_retina_mask(m).astype(bool)
                  & valid_retina_mask(w).astype(bool))
        if common.sum() > 500:
            before_corr = np.corrcoef(f[:,:,1][common], m[:,:,1][common])
            after_corr = np.corrcoef(f[:,:,1][common], w[:,:,1][common])
            print('Matrice de corrélation AVANT :\n', before_corr)
            print('Matrice de corrélation APRÈS :\n', after_corr)
else:
    print('Pas de comparaison photométrique : estimation de transformation échouée.')

## 11 — Flot optique dense de Farnebäck (résiduel)

Le recalage est d'abord réalisé par transformation globale affine, puis un champ `(u,v)` est estimé entre l'image de référence et l'image mobile recalée. Visualisation en HSV : **teinte = direction**, **luminosité = amplitude**. Cette étape est illustrative : les masques de recouvrement et les variations d'intensité limitent la portée anatomique de l'interprétation.

In [ ]:
flow_result = visualize_optical_flow(
    sample, model='affine', max_side=DEMO_SIDE, optical_side=440
)
print('Fin de la démonstration du flot optique résiduel.')

## 12 — Benchmark FIRE complet : 134 paires × 3 modèles

Le programme traite chaque paire **une seule fois pour extraire les primitives**, puis estime successivement Similarité/Affine/Homographie. Les échecs restent enregistrés dans le CSV. `MAX_PAIRS = None` signifie **les 134 paires** (si les fichiers sont présents) ; mets `10` pour un essai rapide. Sur Kaggle CPU, **cette cellule peut durer plusieurs minutes**.

Critère pédagogique de succès : `TRE moyenne < THRESHOLD_PX`, ici 50 pixels de l'image d'origine. Ce n'est **pas l'AUC du protocole officiel FIRE** ; nous ne prétendons pas reproduire son score standardisé.

In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
results = evaluate_dataset(pairs, models=BENCH_MODELS, max_side=BENCH_SIDE,
                           max_pairs=MAX_PAIRS, verbose=True)
summary = summarize_results(results, threshold_px=THRESHOLD_PX)

results.to_csv(RESULT_DIR / 'fire_resultats_par_paire.csv', index=False)
summary.to_csv(RESULT_DIR / 'fire_synthese_par_categorie.csv', index=False)
print('\n=== RÉSUMÉ DU BENCHMARK ===')
display(summary.round(3))
print('Paires analysées :', results['pair_id'].nunique(), '/', len(pairs))
print('Nombre de lignes :', len(results))
print('Succès / échecs de transformation :', results['status'].value_counts().to_dict())
print('CSV disponibles dans :', RESULT_DIR)

## 13 — Figures et analyse des échecs

Le graphe compare le **taux de succès** par catégorie et transformation ; les échecs comptent dans le dénominateur. Nous affichons également les paires **affines** les plus difficiles (échecs d'abord puis grandes TRE). Attention : la moyenne TRE n'est calculée que sur les recalages où une transformation est disponible.

In [ ]:
if not summary.empty:
    piv = summary.pivot(index='category', columns='model', values='success_rate')
    piv = piv.reindex([c for c in ('S','A','P') if c in piv.index])
    ax = piv.plot(kind='bar', figsize=(9,5), ylim=(0,1), rot=0, grid=True)
    ax.set_ylabel('Fraction des paires (échecs inclus)')
    ax.set_xlabel('Catégorie FIRE')
    ax.set_title(f'Succès selon la catégorie — TRE < {THRESHOLD_PX:g} pixels originaux')
    plt.tight_layout()
    plt.savefig(RESULT_DIR/'taux_succes_par_categorie.png', dpi=160, bbox_inches='tight')
    plt.show()

    aff = results[results.model=='affine'].copy()
    aff['non_ok'] = aff.status.ne('ok')
    aff = aff.sort_values(['non_ok','tre_after_px'], ascending=[False,False], na_position='first')
    print('Paires affines problématiques (échecs puis plus fortes TRE) :')
    display(aff[['pair_id','category','status','n_matches','n_inliers','tre_before_px','tre_after_px']].head(12).round(2))
    print('Pour les modèles qui échouent, on ne peut pas interpréter une TRE après recalage.')
else:
    print('Résumé vide : aucun benchmark valide.')

## 14 — Conclusions et réponses pour un compte rendu

**Décrire la chaîne :** extraction de primitives → mise en correspondance → optimisation robuste RANSAC → transformation de l'image mobile → évaluation par landmarks. Les points manuels constituent une référence externe indépendante.

**Comparer les transformations :** un modèle plus flexible peut améliorer un recalage mais aussi produire des résultats aberrants si les correspondances sont mauvaises. Les catégories S et A offrent généralement un fort recouvrement, tandis que les P sont conçues pour tester un faible recouvrement. Il faut comparer **à la fois** la TRE, le taux d'échec et le taux de réussite.

**Discuter les critères :** le MSE/SSD pénalise les différences d'intensité, la NCC est une corrélation linéaire, la NMI se fonde sur les distributions conjointes. Ils complètent mais ne remplacent **pas** la TRE géométrique sur les points annotés.

**Différencier flot optique et recalage rigide :** le flot fournit un champ local dense, mais des différences de texture ou de luminosité peuvent donner l'impression d'une déformation. Ce TP n'évalue pas scientifiquement un algorithme élastique ni un diagnostic médical.

**Limites :** usage de SIFT/ORB et de modèles géométriques globaux ; pas de segmentation vasculaire profonde ni de modèle anatomique dédié ; difficulté particulière des faibles recouvrements ; expérience pédagogique avec un seuil de TRE non officiel.

### Fichiers de sortie

- `/kaggle/working/FIRE_results/fire_resultats_par_paire.csv` : résultats détaillés, erreurs et statuts pour chaque modèle.
- `/kaggle/working/FIRE_results/fire_synthese_par_categorie.csv` : tableau comparatif S/A/P.
- `/kaggle/working/FIRE_results/taux_succes_par_categorie.png` : graphique.

Tu peux les récupérer dans l'onglet **Output** du notebook Kaggle après exécution et sauvegarde d'une version.